# 1.关于模型的profile的使用

举例1：调用deepseek模型

In [ ]:
from sys import int_info

from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv

# 从.env文件种加载环境变量
load_dotenv(override = True)

model = ChatDeepSeek(
    model = "deepseek-flash",
    temperature = 0.8,
    max_tokens = 1000,
    max_retries = 6,
)

print(model.profile)

In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.chat_models import ChatTongyi

# 1、读取.env配置文件中的信息；相关的环境变量要以.env中环境变量优先
load_dotenv(override = True)

DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")

# 2、模型的初始化
llm_tongyi = ChatTongyi(
    model = "qwen3-max",
    api_key = DASHSCOPE_API_KEY,
)

print(llm_tongyi.profile)

# 2.模型初始化完整参数

举例1：

In [ ]:
from langchain_deepseek import ChatDeepSeek

print(ChatDeepSeek.model_fields.keys())

举例2：使用init_chat_model()

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override = True)

temp_model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
)

print(temp_model.model_fields.keys())

举例3：测试model_kwargs的使用

langchain没有列出来的字段但是模型本身支持，那我们可以把参数声明在model_kwargs参数中

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv(override = True)

model = init_chat_model(
    model = "deepseek:deepseek-flash",
    model_kwargs = {"tools": [
        {
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Get weather of a location, the user should supply a location first.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. San Francisco, CA",
                        }
                    },
                    "required": ["location"]
                },
            }
        },
    ]}
)

# res = model.invoke("你好，今天北京天气如何？")
res = model.invoke("1 + 2 = ?")
rprint(res)

举例4：extra_body

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv(override = True)

model = init_chat_model(
    model = "deepseek:deepseek-flash",
    extra_body = {"thinking": {"type": "enabled"}},
)

res = model.invoke("你好，一句话回答")
rprint(res)  # 出现 reasoning_content也就是思考模式下的回答

# 3.模型调用中的config参数(invoke stream batch)

In [13]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 1. 初始化模型
model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    temperature=0.2,
    max_tokens=500,
    # 指定可调整参数
    configurable_fields=("model", "model_provider", "temperature", "max_tokens"),
)

# 2. 准备 config 字典
config = {
    "run_name": "joke_generation",      # 在LangSmith中这次运行会显示为 "joke_generation"
    "tags": ["tag1", "tag2"],           # 打上标签便于分类查找
    "metadata": {"user_id": "123"},     # 记录用户ID
    "configurable":{
        "model": "deepseek-v4-pro",     # 配置模型参数
        "model_provider": "openai",     # 配置模型提供商参数
        "temperature": 0.7,             # 配置温度参数
        "max_tokens": 1000              # 配置最大令牌数
    }
}

# 3. 调用模型并传入config
response = model.invoke(
    "1 + 2 = ？",
    config=config
)

rprint(response)

AIMessage(
    content='3',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'We need answer simple. Need comply. User asks Chinese: 1+2=? Answer 3. final.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 25,
            'prompt_tokens': 37,
            'total_tokens': 62,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 23,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 37
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': 'fb0d9806-7622-497c-a23e-5696a29c77db',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a11e66-8050-7982-9842-4d676390b695-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 37,
        'output_tokens': 25,
        'total_tokens': 62,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 23}
    }
)